# Model Training and Storage to watsonx.ai

This notebook demonstrates the workflow for training a TensorFlow model and storing it to watsonx.ai deployment space.

## Workflow
1. Setup and Configuration
2. Load Training Data from Project Assets
3. Load Pre-trained H5 Model
4. Train/Fine-tune Model with Data
5. Convert to SavedModel Format
6. Store Model to watsonx.ai Deployment Space
7. Deploy Model to watsonx.ai


## 1. Setup and Configuration

In [2]:
# Import required libraries
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
from tensorflow import keras
import tensorflow as tf
from importlib.metadata import version
import warnings
warnings.filterwarnings('ignore')

# Set timezone to UTC+8 (Asia/Taipei)
TZ_UTC8 = timezone(timedelta(hours=8))

# Get package versions
tf_version = version('tensorflow')
np_version = version('numpy')
pd_version = version('pandas')
watsonx_version = version('ibm-watsonx-ai')

print("="*80)
print("PACKAGE VERSIONS")
print("="*80)
print(f"✅ TensorFlow version: {tf_version}")
print(f"✅ NumPy version: {np_version}")
print(f"✅ Pandas version: {pd_version}")
print(f"✅ IBM watsonx.ai version: {watsonx_version}")
print("\n✅ All libraries imported successfully")

PACKAGE VERSIONS
✅ TensorFlow version: 2.14.1
✅ NumPy version: 1.26.4
✅ Pandas version: 2.1.4
✅ IBM watsonx.ai version: 1.5.3

✅ All libraries imported successfully


### Load Environment Variables

This cell loads the required credentials from environment variables.

In [4]:
# Load credentials from environment variables
PROJECT_ID = os.getenv('PROJECT_ID')
SPACE_ID = os.getenv('SPACE_ID')
WX_API_KEY = os.getenv('WX_API_KEY')
MODEL_NAME = os.getenv('MODEL_NAME')

print("="*80)
print("ENVIRONMENT VARIABLES CHECK")
print("="*80)
print(f"PROJECT_ID: {'✅ Set' if PROJECT_ID else '❌ Not set'}")
print(f"SPACE_ID: {'✅ Set' if SPACE_ID else '❌ Not set'}")
print(f"WX_API_KEY: {'✅ Set' if WX_API_KEY else '❌ Not set'}")
print(f"MODEL_NAME: {'✅ Set' if MODEL_NAME else '❌ Not set'}")
print("="*80)

if not all([PROJECT_ID, SPACE_ID, WX_API_KEY, MODEL_NAME]):
    print("\n⚠️ WARNING: Required environment variables are missing!")
    print("Please set: PROJECT_ID, SPACE_ID, WX_API_KEY, MODEL_NAME")
else:
    print("\n✅ All required environment variables are set!")

ENVIRONMENT VARIABLES CHECK
PROJECT_ID: ✅ Set
SPACE_ID: ✅ Set
WX_API_KEY: ✅ Set
MODEL_NAME: ✅ Set

✅ All required environment variables are set!


## 2. Connect to watsonx.ai

#### CP4D

In [ ]:
# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================

username = 'PASTE YOUR USERNAME HERE'
url = 'PASTE THE PLATFORM URL HERE'

print("✅ Configuration loaded")

In [ ]:
from ibm_watsonx_ai import Credentials, APIClient

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.1"
)

client = APIClient(credentials)
client.set.default_project(PROJECT_ID)

print("="*80)
print("watsonx.ai Connection")
print("="*80)
print(f"✅ Connected to: {credentials.url}")
print(f"✅ Project ID: {PROJECT_ID}")
print(f"✅ Client version: {client.version}")

#### IBM Cloud

In [ ]:
# from ibm_watsonx_ai import APIClient
# from ibm_watsonx_ai import Credentials

# # Connect to watsonx.ai
# credentials = Credentials(
#     url="https://us-south.ml.cloud.ibm.com",  # Replace with your region
#     api_key=WX_API_KEY
# )

# client = APIClient(credentials)
# client.set.default_project(PROJECT_ID)

# print("="*80)
# print("watsonx.ai Connection")
# print("="*80)
# print(f"✅ Connected to: {credentials.url}")
# print(f"✅ Project ID: {PROJECT_ID}")
# print(f"✅ Client version: {client.version}")

watsonx.ai Connection
✅ Connected to: https://us-south.ml.cloud.ibm.com
✅ Project ID: c4d2e318-f9d4-47c7-9482-f313463b22ea
✅ Client version: 1.5.3


## 3. Find Latest Training Data in Project Assets

In [6]:
print(f"\n{'='*80}")
print("Step 1: Finding Latest Training Data")
print(f"{'='*80}")

# Get all data assets
assets = client.data_assets.get_details()

# Filter for training data CSV files (matching pattern: model_input_data_* or training_data_*)
training_data_assets = []
for asset in assets['resources']:
    name = asset['metadata']['name']
    if 'training_data_' in name:
        training_data_assets.append({
            'name': name,
            'asset_id': asset['metadata']['asset_id'],
            'created_at': asset['metadata'].get('created_at', '')
        })

if not training_data_assets:
    print("❌ No training data found in project assets!")
    print("Please run the data generation notebook first.")
else:
    # Sort by name (which includes timestamp) to get the latest
    training_data_assets.sort(key=lambda x: x['name'], reverse=True)
    
    print(f"\n✅ Found {len(training_data_assets)} training data asset(s):")
    for i, asset in enumerate(training_data_assets[:5], 1):
        print(f"  {i}. {asset['name']} (ID: {asset['asset_id']})")
    
    # Use latest training data
    selected_asset = training_data_assets[0]
    
    print(f"\n✅ Selected training data: {selected_asset['name']}")
    print(f"✅ Asset ID: {selected_asset['asset_id']}")
    
    training_data_asset_id = selected_asset['asset_id']


Step 1: Finding Latest Training Data

✅ Found 8 training data asset(s):
  1. training_data_20260326_122501 (ID: 8684b78b-f115-40ef-a7a7-deffbc66dd7d)
  2. training_data_20260326_121920 (ID: 342c62b1-ab15-476e-ac0e-ddd9c939f471)
  3. training_data_20260326_114120 (ID: f351d025-5cb6-40d1-b87b-fa198576a4f3)
  4. training_data_20260326_113831 (ID: 5ace751b-086b-4e52-bac9-a98069be9224)
  5. training_data_20260326_095300 (ID: 858192a0-6ab7-41e3-88e5-61ad58408430)

✅ Selected training data: training_data_20260326_122501
✅ Asset ID: 8684b78b-f115-40ef-a7a7-deffbc66dd7d


## 4. Download and Load Training Data

In [7]:
print(f"\n{'='*80}")
print("Step 2: Loading Training Data")
print(f"{'='*80}")

# Download the data asset
import io

# Get the data asset content
data_content = client.data_assets.get_content(training_data_asset_id)

# Load into pandas DataFrame
df_train = pd.read_csv(io.BytesIO(data_content))

print(f"✅ Training data loaded successfully")
print(f"   Shape: {df_train.shape}")
print(f"   Columns: {len(df_train.columns)}")
print(f"\nFirst few rows:")
print(df_train.head())

print(f"\n✅ Data ready for training")


Step 2: Loading Training Data
✅ Training data loaded successfully
   Shape: (100, 32)
   Columns: 32

First few rows:
                           input_x_o_MD_BRN_CONTACTS  \
0  [0.4967141530112327, -0.13826430117118466, 0.6...   
1  [0.25739992534469336, -0.9084814327806611, -0....   
2  [-0.7506147172558728, 1.3163573247118194, 1.24...   
3  [0.026374772849561846, 0.26032170142265076, -0...   
4  [0.5848758398705177, 1.2311957404405667, 0.821...   

  input_x_c_1_MD_BRN_CONTACTS input_x_c_2_MD_BRN_CONTACTS  \
0                        [10]                         [1]   
1                         [9]                         [5]   
2                        [10]                         [2]   
3                         [4]                         [1]   
4                         [0]                         [2]   

  input_x_c_3_MD_BRN_CONTACTS input_x_c_4_MD_BRN_CONTACTS  \
0                         [2]                         [5]   
1                         [0]                         [

## 5. Load Pre-trained H5 Model from Project Assets

**Note**: Use the "Code Snippets" feature to load your H5 model, or we'll create a simple demonstration model.

In [8]:
print(f"\n{'='*80}")
print("Step 3: Loading Pre-trained H5 Model")
print(f"{'='*80}")

# Find init_weight.h5 in project assets
print("\nSearching for init_weight.h5 in project assets...")
assets = client.data_assets.get_details()

h5_model_asset = None
for asset in assets['resources']:
    name = asset['metadata']['name']
    if 'init_weight.h5' in name or name == 'init_weight.h5':
        h5_model_asset = asset
        break

if h5_model_asset:
    h5_asset_id = h5_model_asset['metadata']['asset_id']
    h5_asset_name = h5_model_asset['metadata']['name']
    print(f"✅ Found H5 model: {h5_asset_name}")
    print(f"   Asset ID: {h5_asset_id}")
    
    # Download and load the H5 model
    print(f"\nDownloading H5 model...")
    h5_model_content = client.data_assets.get_content(h5_asset_id)
    
    # Save temporarily
    temp_h5_path = 'temp_init_weight.h5'
    with open(temp_h5_path, 'wb') as f:
        f.write(h5_model_content)
    print(f"✅ H5 model downloaded: {len(h5_model_content) / (1024*1024):.2f} MB")
    
    # Load the model
    print(f"\nLoading H5 model into memory...")
    model = keras.models.load_model(temp_h5_path)
    print(f"✅ Model loaded successfully!")
    
    # Clean up temporary file
    import os as os_module
    os_module.remove(temp_h5_path)
    print(f"✅ Temporary file cleaned up")
else:
    print("❌ init_weight.h5 not found in project assets!")
    print("Please upload init_weight.h5 to project assets first.")
    raise FileNotFoundError("init_weight.h5 not found in project assets")

# Display model information
print(f"\n📊 Model Information:")
print(f"   Model name: {model.name}")
print(f"   Total layers: {len(model.layers)}")
print(f"   Inputs: {len(model.inputs)}")
print(f"   Outputs: {len(model.outputs)}")
print(f"   Total parameters: {model.count_params():,}")

# Display input/output details
print(f"\n📥 Model Inputs:")
for i, inp in enumerate(model.inputs[:5], 1):
    print(f"   {i}. {inp.name}: {inp.shape}")
if len(model.inputs) > 5:
    print(f"   ... and {len(model.inputs) - 5} more inputs")

print(f"\n📤 Model Outputs:")
for i, out in enumerate(model.outputs, 1):
    print(f"   {i}. {out.name}: {out.shape}")


Step 3: Loading Pre-trained H5 Model

Searching for init_weight.h5 in project assets...
✅ Found H5 model: init_weight.h5
   Asset ID: 380ab5e7-ea2b-4dd1-808b-e4e83ee46f3f

✅ H5 model downloaded: 5.24 MB

Loading H5 model into memory...
✅ Model loaded successfully!
✅ Temporary file cleaned up

📊 Model Information:
   Model name: buy_model_exchange
   Total layers: 45
   Inputs: 32
   Outputs: 3
   Total parameters: 1,315,363

📥 Model Inputs:
   1. input_x_o_MD_BRN_CONTACTS: (None, 16)
   2. input_x_c_1_MD_BRN_CONTACTS: (None, 1)
   3. input_x_c_2_MD_BRN_CONTACTS: (None, 1)
   4. input_x_c_3_MD_BRN_CONTACTS: (None, 1)
   5. input_x_c_4_MD_BRN_CONTACTS: (None, 1)
   ... and 27 more inputs

📤 Model Outputs:
   1. y_layers/buy_19/Sigmoid:0: (None, 19)
   2. y_layers_2/exchange_9/Sigmoid:0: (None, 9)
   3. y_layers/order_6/Softmax:0: (None, 6)


## 6. Compile and Train Model

In [9]:
print(f"\n{'='*80}")
print("Step 4: Training Model")
print(f"{'='*80}")

# Compile model
model.compile(
    optimizer='adam',
    loss=['categorical_crossentropy', 'categorical_crossentropy', 'categorical_crossentropy'],
    metrics=['accuracy']
)

print("✅ Model compiled")

# Prepare training data
# Note: This is a simplified example. In production, you would:
# 1. Parse the CSV data properly
# 2. Create proper input arrays for each of the 32 inputs
# 3. Create proper output labels for the 3 outputs

print("\n⚠️ Note: Actual training requires proper data preprocessing")
print("This is a demonstration of the workflow.")
print("In production, implement proper data preprocessing based on your data format.")

# For demonstration, we'll skip actual training
# Uncomment the following to train:
"""
# Prepare X_train (list of 32 input arrays) and y_train (list of 3 output arrays)
# X_train = [...]  # 32 arrays
# y_train = [...]  # 3 arrays

# Train model
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)
"""

print("\n✅ Model ready (training skipped for demonstration)")


Step 4: Training Model
✅ Model compiled

⚠️ Note: Actual training requires proper data preprocessing
This is a demonstration of the workflow.
In production, implement proper data preprocessing based on your data format.

✅ Model ready (training skipped for demonstration)


## 7. Convert to SavedModel Format

In [10]:
print(f"\n{'='*80}")
print("Step 5: Converting to SavedModel Format")
print(f"{'='*80}")

# Use model name from environment variable
output_dir = MODEL_NAME

# Save as SavedModel
model.save(output_dir, save_format='tf')
print(f"✅ Model saved to: {output_dir}")

# Create compressed archive
import subprocess
zip_file = f"{output_dir}.zip"

result = subprocess.run(
    ['sh', '-c', f'cd {output_dir} && zip -r -9 ../{os.path.basename(zip_file)} .'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    zip_size = os.path.getsize(zip_file)
    print(f"✅ Created compressed archive: {zip_file}")
    print(f"✅ Archive size: {zip_size / (1024*1024):.2f} MB")
else:
    print(f"✗ Failed to create archive: {result.stderr}")


Step 5: Converting to SavedModel Format
INFO:tensorflow:Assets written to: savedmodel_1231/assets


INFO:tensorflow:Assets written to: savedmodel_1231/assets


✅ Model saved to: savedmodel_1231
✅ Created compressed archive: savedmodel_1231.zip
✅ Archive size: 5.09 MB


## 8. Store Model to watsonx.ai Deployment Space

In [ ]:
print(f"\n{'='*80}")
print("Step 6: Storing Model to watsonx.ai Deployment Space")
print(f"{'='*80}")

# Set deployment space
client.set.default_space(SPACE_ID)
print(f"✅ Using deployment space: {SPACE_ID}")

# Store model
model_name = MODEL_NAME

model_meta_props = {
    client.repository.ModelMetaNames.NAME: model_name,
    client.repository.ModelMetaNames.TYPE: "tensorflow_2.14",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_UID: client.software_specifications.get_id_by_name("runtime-24.1-py3.11")
}

print(f"\n📦 Storing model: {model_name}")
published_model = client.repository.store_model(
    model=zip_file,
    meta_props=model_meta_props
)

model_uid = client.repository.get_model_id(published_model)
print(f"✅ Model stored successfully!")
print(f"✅ Model ID: {model_uid}")
print(f"✅ Model Name: {model_name}")


Step 6: Storing Model to watsonx.ai Deployment Space
Unsetting the project_id ...
✅ Using deployment space: 165e735f-11f6-4879-a6fd-99cfecf44f37

📦 Storing model: savedmodel_1231
✅ Model stored successfully!
✅ Model ID: c681fc47-d9cf-495f-971d-4891150d574b
✅ Model Name: savedmodel_1231


## 9. Deploy Model to watsonx.ai

In [14]:
print(f"\n{'='*80}")
print("Step 7: Deploying Model to watsonx.ai")
print(f"{'='*80}")

# Check if deployment already exists
print(f"\nChecking for existing deployments...")
deployments = client.deployments.get_details()
existing_deployment_id = None
existing_deployment_name = None

for deployment in deployments['resources']:
    print(deployment)
    deployment_name = deployment['metadata']['name']
    deployment_asset = deployment['entity'].get('asset', {})
    deployment_model_id = deployment_asset.get('id', '')
    
    # Check if this deployment is for our model
    if deployment_model_id == model_uid or f"{MODEL_NAME}_deployment" == deployment_name:
        existing_deployment_id = deployment['metadata']['id']
        existing_deployment_name = deployment_name
        print(f"✅ Found existing deployment: {deployment_name}")
        print(f"   Deployment ID: {existing_deployment_id}")
        break

if existing_deployment_id:
    print(f"\n⚠️ Deployment already exists. Updating...")
    
    # Delete old deployment
    client.deployments.delete(existing_deployment_id)
    print(f"✅ Deleted old deployment: {existing_deployment_name}")

# Create new deployment
deployment_name = f"{MODEL_NAME}_deployment"

deployment_meta_props = {
    client.deployments.ConfigurationMetaNames.NAME: deployment_name,
    client.deployments.ConfigurationMetaNames.ONLINE: {},
}

print(f"\n🚀 Creating deployment: {deployment_name}")
deployment = client.deployments.create(
    model_uid,
    meta_props=deployment_meta_props
)

deployment_id = client.deployments.get_id(deployment)
print(f"✅ Deployment created successfully!")
print(f"✅ Deployment ID: {deployment_id}")
print(f"✅ Deployment Name: {deployment_name}")

# Wait for deployment to be ready
print(f"\n⏳ Waiting for deployment to be ready...")
import time
max_wait = 300  # 5 minutes
wait_interval = 10  # 10 seconds
elapsed = 0

while elapsed < max_wait:
    deployment_details = client.deployments.get_details(deployment_id)
    status = deployment_details['entity']['status']['state']
    
    if status == 'ready':
        print(f"✅ Deployment is ready!")
        break
    elif status == 'failed':
        print(f"❌ Deployment failed!")
        print(f"   Error: {deployment_details['entity']['status'].get('message', 'Unknown error')}")
        break
    else:
        print(f"   Status: {status} (waited {elapsed}s)")
        time.sleep(wait_interval)
        elapsed += wait_interval

if elapsed >= max_wait:
    print(f"⚠️ Deployment timeout after {max_wait}s")
    print(f"   Please check deployment status manually")


Step 7: Deploying Model to watsonx.ai

Checking for existing deployments...
{'entity': {'asset': {'id': '2be956b1-601c-4a62-9de6-8c3f930962ce'}, 'chat_enabled': False, 'custom': {}, 'deployed_asset_type': 'model', 'hardware_spec': {'name': 'XXS', 'num_nodes': 1}, 'online': {}, 'status': {'inference': [{'url': 'https://us-south.ml.cloud.ibm.com/ml/v4/deployments/569c8031-0f77-4914-acb2-0d3748d62b7e/predictions'}], 'online_url': {'url': 'https://us-south.ml.cloud.ibm.com/ml/v4/deployments/569c8031-0f77-4914-acb2-0d3748d62b7e/predictions'}, 'serving_urls': ['https://us-south.ml.cloud.ibm.com/ml/v4/deployments/569c8031-0f77-4914-acb2-0d3748d62b7e/predictions'], 'state': 'ready'}, 'tooling': {}}, 'metadata': {'created_at': '2026-03-26T03:07:52.350Z', 'id': '569c8031-0f77-4914-acb2-0d3748d62b7e', 'modified_at': '2026-03-26T03:07:52.350Z', 'name': 'SavedModel_20260326_104724_deployment', 'owner': 'IBMid-694000GICY', 'space_id': '165e735f-11f6-4879-a6fd-99cfecf44f37'}, 'system': {'warnings': 

## 10. Summary

In [15]:
print(f"\n{'='*80}")
print("MODEL TRAINING AND STORAGE SUMMARY")
print(f"{'='*80}")

print(f"\n✅ Training Data:")
print(f"   - Asset ID: {training_data_asset_id}")
print(f"   - Samples: {len(df_train)}")
print(f"   - Features: {len(df_train.columns)}")

print(f"\n✅ Model Information:")
print(f"   - Model Name: {model_name}")
print(f"   - Model ID: {model_uid}")
print(f"   - Total Parameters: {model.count_params():,}")
print(f"   - Inputs: {len(model.inputs)}")
print(f"   - Outputs: {len(model.outputs)}")

print(f"\n✅ Storage Information:")
print(f"   - Deployment Space ID: {SPACE_ID}")
print(f"   - SavedModel Directory: {output_dir}")
print(f"   - Compressed Archive: {zip_file}")
print(f"   - Archive Size: {zip_size / (1024*1024):.2f} MB")

print(f"\n✅ Deployment Information:")
print(f"   - Deployment ID: {deployment_id}")
print(f"   - Deployment Name: {deployment_name}")
print(f"   - Deployment Status: ready")

print(f"\n{'='*80}")
print("✅ Model Training and Storage Complete!")
print(f"{'='*80}")
print(f"\n📝 Next Steps:")
print(f"   - Use notebook 3 to deploy and test this model")
print(f"   - Model ID for deployment: {model_uid}")


MODEL TRAINING AND STORAGE SUMMARY

✅ Training Data:
   - Asset ID: 8684b78b-f115-40ef-a7a7-deffbc66dd7d
   - Samples: 100
   - Features: 32

✅ Model Information:
   - Model Name: savedmodel_1231
   - Model ID: c681fc47-d9cf-495f-971d-4891150d574b
   - Total Parameters: 1,315,363
   - Inputs: 32
   - Outputs: 3

✅ Storage Information:
   - Deployment Space ID: 165e735f-11f6-4879-a6fd-99cfecf44f37
   - SavedModel Directory: savedmodel_1231
   - Compressed Archive: savedmodel_1231.zip
   - Archive Size: 5.09 MB

✅ Deployment Information:
   - Deployment ID: 84716fe3-b335-43c1-8411-5a8ce49e4486
   - Deployment Name: savedmodel_1231_deployment
   - Deployment Status: ready

✅ Model Training and Storage Complete!

📝 Next Steps:
   - Use notebook 3 to deploy and test this model
   - Model ID for deployment: c681fc47-d9cf-495f-971d-4891150d574b


## Summary

This notebook has demonstrated:
1. ✅ Loading training data from watsonx.ai project assets
2. ✅ Loading/creating a TensorFlow model
3. ✅ Training the model (workflow shown, actual training requires proper data preprocessing)
4. ✅ Converting to SavedModel format
5. ✅ Storing model to watsonx.ai deployment space as an asset
6. ✅ Deploying model to watsonx.ai

**Next Steps:**
- Run notebook 3 to deploy and test the stored model
- Implement proper data preprocessing for your specific use case
- Load your actual pre-trained H5 model
- Perform actual model training/fine-tuning